# Lesson 3 · Phase 2 — Reasoning Prompts: CoT, Self-Consistency & Tree of Thoughts

**Mastering Agentic AI Certification · Pre-read**

> Hard problems fail when the model blurts an answer in one step. These techniques give the model **room to think** — first along one path, then across many, then by deliberate search.

## 1. The complete picture — three levels of "thinking"

```
 CHAIN OF THOUGHT        SELF-CONSISTENCY              TREE OF THOUGHTS
   one path                many paths, vote              branch · evaluate · search

   Q                          Q                                Q
   │                       ┌──┼──┐                          ┌──┴──┐
  step                    p1 p2 p3   (sample N)            b1     b2     (expand)
   │                       │  │  │                        ┌┴┐    ┌┴┐
  step                    a1 a2 a1                        ..  prune bad branches
   │                        └──┬──┘                         keep promising ones
  ANSWER                  majority → a1                    best leaf → ANSWER
```

More reliability ↑, more tokens/cost ↑ as you move right. Pick the cheapest one that's reliable enough.

## 2. Chain of Thought (CoT)

Ask the model to **show its reasoning step by step** before the final answer. The intermediate steps act as a scratchpad that makes each sub-step easier and the final answer more accurate.

```
Q: A shop had 23 apples, sold 7 in the morning and 9 in the afternoon, then
   received a delivery of 15. How many now?
A: Let's think step by step.
   Start: 23. Sold 7 → 16. Sold 9 → 7. Delivery +15 → 22.
   Final answer: 22.
```

- Trigger phrases: *"Let's think step by step."*, *"Show your reasoning."*
- ✅ Big gains on math, logic, multi-step questions.
- ⚠️ More tokens; reasoning can still be wrong (it's not a proof).

## 3. Self-Consistency

CoT picks **one** reasoning path, which might be a fluke. **Self-consistency** samples **several** independent CoT paths (higher temperature) and takes a **majority vote** on the final answer. Correct reasoning tends to agree; mistakes scatter.

```
sample 5 chains → answers: [22, 22, 24, 22, 7]  →  majority vote → 22
```

- ✅ Noticeably more robust than a single chain.
- ⚠️ N× the cost (you run the model N times).

## 4. Tree of Thoughts (ToT)

Instead of one linear chain, the model **branches** into multiple partial thoughts, **evaluates** how promising each is, **prunes** weak branches, and **explores** the good ones (BFS/DFS-style search). It's deliberate problem-solving, not a single guess.

```
            problem
           /        \
      approach A    approach B
       /    \          (score low → prune)
   step    step
   (score, keep best) → continue → solution
```

- ✅ Best for puzzles, planning, search-like problems where you must try and backtrack.
- ⚠️ Most expensive; needs an evaluation step and orchestration code.

In [ ]:
# SELF-CONSISTENCY demo: several reasoning paths, majority vote.
# The 'paths' are simulated CoT outputs (some correct, some not).
from collections import Counter

def solve_apples():
    # the deterministic correct chain
    n = 23; n -= 7; n -= 9; n += 15
    return n

# Simulate 5 sampled chains: most get it right, a couple slip (as real sampling does).
sampled_answers = [22, 22, 24, 22, 7]   # stand-in for 5 stochastic CoT runs
vote = Counter(sampled_answers).most_common(1)[0]
print("deterministic CoT answer :", solve_apples())
print("5 sampled chain answers  :", sampled_answers)
print(f"self-consistency vote    : {vote[0]}  (chosen by {vote[1]}/5 paths)")

In [ ]:
# TREE OF THOUGHTS demo (toy): reach a target number from a start using +ops.
# We BRANCH over candidate steps, SCORE each by closeness, PRUNE weak branches,
# and keep exploring the promising ones until one branch hits the target.
def tot_search(start, target, ops=(("+5",5),("+9",9),("-7",-7),("+15",15),("-9",-9)),
               beam=2, depth=4):
    frontier = [(start, [str(start)])]          # each item: (value, path-of-steps)
    for d in range(depth):
        children = []
        for val, path in frontier:
            for name, delta in ops:             # BRANCH: every op from every kept node
                children.append((val + delta, path + [name]))
        children.sort(key=lambda vp: abs(vp[0] - target))   # EVALUATE by closeness
        frontier = children[:beam]                          # PRUNE to the 'beam' best
        print(f"depth {d+1}: kept {[ (v, ' '.join(p)) for v,p in frontier]}")
        if frontier[0][0] == target:            # SUCCESS: a branch reached the goal
            return frontier[0]
    return frontier[0]

print("Goal: from 10 reach 24\n")
val, path = tot_search(10, 24)
status = "REACHED TARGET" if val == 24 else "best effort"
print(f"\nBest branch ({status}):", val, "via", " ".join(path))

## 5. Comparison — reliability vs cost

| Technique | Idea | Reliability | Relative cost |
|---|---|---|---|
| **Chain of Thought** | one step-by-step path | ↑ over direct answer | 1× |
| **Self-Consistency** | N CoT paths + majority vote | ↑↑ | N× |
| **Tree of Thoughts** | branch, score, prune, search | ↑↑↑ (search problems) | high (many calls + control logic) |

Rule of thumb: **direct → CoT → self-consistency → ToT**, escalating only when accuracy demands it.

## 6. How this contributes to agentic AI

- An agent must **plan** multi-step tasks — CoT is the simplest planning prompt.
- **Self-consistency** lets an agent double-check critical decisions (e.g. "should I delete this file?") by voting across samples.
- **Tree of Thoughts** is the prompting ancestor of agent **search/planning**: propose options, evaluate, backtrack — the core loop of deliberate agents.
- These feed directly into **Phase 3**, where reasoning is combined with **taking actions**.

## 7. Key takeaways

1. **Chain of Thought** = make the model show step-by-step reasoning → fewer slips on multi-step tasks.
2. **Self-Consistency** = sample many CoT paths and **majority-vote** → robustness at N× cost.
3. **Tree of Thoughts** = **branch, evaluate, prune, search** → best for puzzle/planning problems.
4. Reliability and cost both rise left→right; escalate only as needed.
5. These reasoning patterns are the bridge to **agentic** behaviour (Phase 3).

---
*Next (Phase 3):* combining reasoning with action — **ReAct** and **Prompt Chaining**.

In [ ]:
import sys, platform
print("Python :", sys.version.split()[0]); print("Platform:", platform.platform())
print("Lesson 3 · Phase 2 (CoT, Self-Consistency, ToT) notebook is running ✓")